In [21]:
# Current data fromat 
# {"messages": [
#     {"role": "system", "content": "You are a licensing compliance expert specifically for university and Greek organization apparel. Your task is to evaluate designs against the established licensing guidelines of these specific organizations. Determine if a design meets all requirements or violates any rules, assuming proper licensing permissions are already in place. For each evaluation, you must respond in a strict two-line format: first indicating 'Compliance Status: Compliant' or 'Compliance Status: Non-compliant', followed by 'Violation Reason:' with either 'None' for compliant designs or a brief explanation for non-compliant designs. Never elaborate beyond this format. Base your evaluation solely on actual violations present in the image, not hypothetical concerns."}, 
#     {"role": "user", "content": [
#             {"type": "text", "text": "Review this apparel design for compliance with licensing rules. Provide compliance status and violation reason, if any."}, 
#             {"type": "image_url", "image_url": {"url": "https://cf.freshprints.com/designs/1650260627094_front.jpeg"}}, 
#             {"type": "image_url", "image_url": {"url": "https://cf.freshprints.com/designs/1650260627094_back.jpeg"}}
#         ]
#     }, 
#     {"role": "assistant", "content": "Compliance Status: Compliant\nViolation Reason: None"}
# ]}
# {"messages": [{"role": "system", "content": "You are a licensing compliance expert specifically for university and Greek organization apparel. Your task is to evaluate designs against the established licensing guidelines of these specific organizations. Determine if a design meets all requirements or violates any rules, assuming proper licensing permissions are already in place. For each evaluation, you must respond in a strict two-line format: first indicating 'Compliance Status: Compliant' or 'Compliance Status: Non-compliant', followed by 'Violation Reason:' with either 'None' for compliant designs or a brief explanation for non-compliant designs. Never elaborate beyond this format. Base your evaluation solely on actual violations present in the image, not hypothetical concerns."}, {"role": "user", "content": [{"type": "text", "text": "Review this apparel design for compliance with licensing rules. Provide compliance status and violation reason, if any."}, {"type": "image_url", "image_url": {"url": "https://cf.freshprints.com/designs/1650136241773_front.jpeg"}}, {"type": "image_url", "image_url": {"url": "https://cf.freshprints.com/designs/1650136241773_back.jpeg"}}]}, {"role": "assistant", "content": "Compliance Status: Compliant\nViolation Reason: None"}]}
# {"messages": [{"role": "system", "content": "You are a licensing compliance expert specifically for university and Greek organization apparel. Your task is to evaluate designs against the established licensing guidelines of these specific organizations. Determine if a design meets all requirements or violates any rules, assuming proper licensing permissions are already in place. For each evaluation, you must respond in a strict two-line format: first indicating 'Compliance Status: Compliant' or 'Compliance Status: Non-compliant', followed by 'Violation Reason:' with either 'None' for compliant designs or a brief explanation for non-compliant designs. Never elaborate beyond this format. Base your evaluation solely on actual violations present in the image, not hypothetical concerns."}, {"role": "user", "content": [{"type": "text", "text": "Review this apparel design for compliance with licensing rules. Provide compliance status and violation reason, if any."}, {"type": "image_url", "image_url": {"url": "https://cf.freshprints.com/designs/1650135259441_front.jpeg"}}, {"type": "image_url", "image_url": {"url": "https://cf.freshprints.com/designs/1650135259441_back.jpeg"}}]}, {"role": "assistant", "content": "Compliance Status: Compliant\nViolation Reason: None"}]}


### Extracting necessary data

In [6]:
import pandas as pd

path = "compliance_verification.jsonl"
# Create DataFrame
df = pd.read_json(path, lines=True)

def extract_image_urls(row):
    images = []
    for msg in row['messages']:
        if msg['role'] == 'user' and isinstance(msg['content'], list):
            for content in msg['content']:
                if 'image_url' in content:
                    images.append(content['image_url']['url'])
    return images #", ".join(images)

# Extract detection status and organization
def extract_info(row):
    for msg in row['messages']:
        if msg['role'] == 'assistant':
            detected = msg['content'].split('\n')[0].split(': ')[1]
            organization = msg['content'].split('\n')[1].split(': ')[1]
            return pd.Series([detected, organization])
    return pd.Series([None, None])


df['image_urls'] = df.apply(extract_image_urls, axis=1)
df[['compliance_status', 'violation_reason']] = df.apply(extract_info, axis=1)

### Data analysis

In [7]:
df

,messages,image_urls,compliance_status,violation_reason
0,"[{'role': 'system', 'content': 'You are a lice...",[https://cf.freshprints.com/designs/1650260627...,Compliant,None
1,"[{'role': 'system', 'content': 'You are a lice...",[https://cf.freshprints.com/designs/1650136241...,Compliant,None
2,"[{'role': 'system', 'content': 'You are a lice...",[https://cf.freshprints.com/designs/1650135259...,Compliant,None
3,"[{'role': 'system', 'content': 'You are a lice...",[https://cf.freshprints.com/designs/1643236625...,Compliant,None
4,"[{'role': 'system', 'content': 'You are a lice...",[https://cf.freshprints.com/designs/1643236624...,Compliant,None
...,...,...,...,...
144173,"[{'role': 'system', 'content': 'You are a lice...",[https://res.cloudinary.com/dsnai3oon/image/up...,Non-compliant,The goat is a trademarked logo and cannot be a...
144174,"[{'role': 'system', 'content': 'You are a lice...",[https://res.cloudinary.com/dsnai3oon/image/up...,Non-compliant,I believe this is still being reworked with a ...
144175,"[{'role': 'system', 'content': 'You are a lice...",[https://res.cloudinary.com/dsnai3oon/image/up...,Non-compliant,WPI as you show it is a trademarked institutio...
144176,"[{'role': 'system', 'content': 'You are a lice...",[https://res.cloudinary.com/dsnai3oon/image/up...,Non-compliant,WPI as you show it is a trademarked institutio...


In [8]:
max(df['image_urls'].apply(lambda x: len(x))),min(df['image_urls'].apply(lambda x: len(x)))

(2, 1)

In [9]:
df['image_urls'].apply(lambda x: len(x)>1).sum()

109150

In [10]:
max(df['violation_reason'].apply(lambda x: len(x))), min(df['violation_reason'].apply(lambda x: len(x)))

(1422, 0)

In [11]:
df['violation_reason'].apply(lambda x: len(x)).describe()

count    144178.000000
mean         14.989867
std          50.859830
min           0.000000
25%           4.000000
50%           4.000000
75%           4.000000
max        1422.000000
Name: violation_reason, dtype: float64

In [12]:
df[df['violation_reason']=='']

,messages,image_urls,compliance_status,violation_reason
143087,"[{'role': 'system', 'content': 'You are a lice...",[https://res.cloudinary.com/dsnai3oon/image/up...,Non-compliant,


In [13]:
df[df['violation_reason']=='']['violation_reason'] = "None"

/tmp/ipykernel_39445/1881144248.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[df['violation_reason']=='']['violation_reason'] = "None"


In [14]:
df['compliance_status'].value_counts()


compliance_status
Compliant        134436
Non-compliant      9742
Name: count, dtype: int64

### Balancing dataset

In [15]:
df_grouped_by = df.groupby(['compliance_status'])
df_balanced = df_grouped_by.apply(lambda x: x.sample(df_grouped_by.size().min()).reset_index(drop=True), include_groups=True).droplevel(1).reset_index(drop=True)
df_balanced['compliance_status'].value_counts()

/tmp/ipykernel_39445/3776268808.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df_grouped_by.apply(lambda x: x.sample(df_grouped_by.size().min()).reset_index(drop=True), include_groups=True).droplevel(1).reset_index(drop=True)


compliance_status
Compliant        9742
Non-compliant    9742
Name: count, dtype: int64

### Creating huggingface dataset

In [17]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df_balanced, test_size=0.2, random_state=42, stratify=df_balanced['compliance_status'])
val_df, test_df = train_test_split(test_df, test_size=0.5, random_state=42, stratify=test_df['compliance_status'])

In [19]:
from datasets import Dataset, DatasetDict, Features, Image, Value
import requests
from PIL import Image as PILImage
from io import BytesIO


def create_split_dataset(df):

    return Dataset.from_dict({
        'image_urls': df['image_urls'],
        'compliance_status': df['compliance_status'],
        'violation_reason': df['violation_reason']
    }, features=Features({
        'image_urls': [Value('string')],  # Supports multiple images per sample
        'compliance_status': Value('string'),
        'violation_reason': Value('string')
    }))

# Create train and test datasets
train_dataset = create_split_dataset(train_df)
test_dataset = create_split_dataset(test_df)
val_dataset = create_split_dataset(val_df)





# # Optimized image loading function
# def load_image(examples):
#     images = []
#     for url in examples['image_urls']:
#         try:
#             response = requests.get(url, timeout=10)
#             response.raise_for_status() # Ensure download was successful
#             images.append(response.content)
#         except requests.RequestException:
#             print(f"Failed to load image from {url}")
#             images.append(None)  # Handle broken URLs
#     return {'images': images}

# # Process dataset with parallel loading
# train_dataset = train_dataset.map(
#     load_image,
#     batched=True,
#     batch_size=100,
#     num_proc=4,  # Number of CPU cores
#     remove_columns=['image_urls'],
#     features=train_dataset.features.copy().update({'images': Image()})
# )

# test_dataset = test_dataset.map(
#     load_image,
#     batched=True,
#     batch_size=100,
#     num_proc=4,  # Number of CPU cores
#     remove_columns=['image_urls'],
#     features=test_dataset.features.copy().update({'images': Image()})
# )


# Create DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset,
    'val': val_dataset
})

In [20]:
dataset

DatasetDict({
    train: Dataset({
        features: ['image_urls', 'compliance_status', 'violation_reason'],
        num_rows: 15587
    })
    test: Dataset({
        features: ['image_urls', 'compliance_status', 'violation_reason'],
        num_rows: 1949
    })
    val: Dataset({
        features: ['image_urls', 'compliance_status', 'violation_reason'],
        num_rows: 1948
    })
})

In [21]:
dataset['train'][0]

{'image_urls': ['https://res.cloudinary.com/dsnai3oon/image/upload/v1740658670/cropped_images/9a3003a295bf13d10d2e1f0f7091e245_1740658669.jpg'],
 'compliance_status': 'Non-compliant',
 'violation_reason': 'wrong university'}

In [22]:
from huggingface_hub import login
import os
# Login into Hugging Face Hub
login(os.getenv("HF_TOKEN"))
dataset.push_to_hub("ikram98ai/compliance_verification")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/16 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/ikram98ai/compliance_verification/commit/44f4f299dc60638c8159cf5d7fafd3dd65684d44', commit_message='Upload dataset', commit_description='', oid='44f4f299dc60638c8159cf5d7fafd3dd65684d44', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ikram98ai/compliance_verification', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ikram98ai/compliance_verification'), pr_revision=None, pr_num=None)

In [16]:
system_prompt = """You are a licensing compliance expert specifically for university and Greek organization apparel. 
Your task is to evaluate designs against the established licensing guidelines of these specific organizations. Determine 
if a design meets all requirements or violates any rules, assuming proper licensing permissions are already in place. 
For each evaluation, you must respond in a strict two-line format: first indicating 'Compliance Status: Compliant' or 
'Compliance Status: Non-compliant', followed by 'Violation Reason:' with either 'None' for compliant designs or a brief 
explanation for non-compliant designs. Never elaborate beyond this format. Base your evaluation solely on actual violations 
present in the image, not hypothetical concerns."""

instruction = """Review this apparel design for compliance with licensing rules. Provide compliance status and violation reason, if any."""

In [17]:
from datasets import load_dataset

train_dataset = load_dataset("ikram98ai/compliance_verification", split="train")
test_dataset = load_dataset("ikram98ai/compliance_verification", split="test")

README.md:   0%|          | 0.00/469 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/995k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/248k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15587 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3897 [00:00<?, ? examples/s]

In [18]:
train_dataset

Dataset({
    features: ['image_urls', 'compliance_status', 'violation_reason'],
    num_rows: 15587
})

To format the dataset, all vision finetuning tasks should be formatted as follows:

```python
[
{ "role": "user",
  "content": [{"type": "text",  "text": instruction}, {"type": "image", "image": image} ]
},
{ "role": "assistant",
  "content": [{"type": "text",  "text": answer} ]
},
]
```

In [19]:

def convert_to_conversation(sample):
    conversation = [
        { "role": "user",
          "content" : [
            {"type" : "text",  "text"  : system_prompt + "\n\n" + instruction},
            ] + [{"type" : "image", "image" : image_url} for image_url in sample["image_urls"]]
        },
        { "role" : "assistant",
          "content" : [
            {"type" : "text",  "text"  : f"Compliance Status: {sample['compliance_status']}\nViolation Reason: {sample['violation_reason']}"} ]
        },
    ]
    return { "messages" : conversation }

In [20]:
con_train_dataset = [convert_to_conversation(sample) for sample in train_dataset]
con_test_dataset = [convert_to_conversation(sample) for sample in test_dataset]